In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
path = str(Path.cwd().parent)
print(path)
sys.path.insert(1, path)

import numpy as np
import pandas as pd
import skforecast

print(skforecast.__version__)

/home/joaquin/Documents/GitHub/skforecast
0.22.0


In [11]:
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 1. Create a dummy NumPy array (X: features, y: target)
# We use float32 for memory efficiency and speed
X = np.random.rand(1000, 10).astype(np.float32)
X[:, -1] = np.random.randint(0, 5, 1000)  # Last column has categories 0, 1, 2, 3, 4
y = np.random.rand(1000)

X = X.astype(object)
X[:, -1] = X[:, -1].astype(int)  # Ensure the categorical column is of integer type

# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize the CatBoostRegressor using the Scikit-Learn API
model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    loss_function='RMSE',
    cat_features=[X.shape[1] - 1],
    verbose=0
)

# 4. Train the model
# Passing a NumPy array directly works perfectly
model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)

# 5. Predict and Evaluate
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"\nFinal RMSE: {rmse:.4f}")


Final RMSE: 0.2847


In [6]:
from catboost import Pool

train_data = Pool(
    data=X_train, 
    label=y_train, 
    cat_features=[9] # Index of the categorical column
)

model = CatBoostRegressor(iterations=100)
model.fit(train_data)

CatBoostError: 'data' is numpy array of floating point numerical type, it means no categorical features, but 'cat_features' parameter specifies nonzero number of categorical features

In [16]:
import numpy as np
import time
from catboost import CatBoostRegressor

# 1. Generate a dataset large enough to see a difference
n_rows = 10_000
n_cols = 10
X_base = np.random.rand(n_rows, n_cols).astype(np.float32)
y = np.random.rand(n_rows)

# Create two versions of the data
# A: Standard Float array (No Categorical)
X_float = X_base.copy()

# B: Object array (Necessary for CatBoost if we flag a column as categorical)
X_obj = X_base.copy().astype(object)

# convert last column to integers for categorical feature
X_obj[:, -1] = X_obj[:, -1].astype(int)

def benchmark_catboost(data, label, is_object=False):
    
    model = CatBoostRegressor(
        iterations=100, 
        verbose=False, 
        thread_count=-1
    )
    
    start_time = time.time()
    model.fit(data, label)
    end_time = time.time()
    
    return end_time - start_time

print("Starting Benchmark...")

# Run float benchmark
time_float = benchmark_catboost(X_float, y, is_object=False)
print(f"Time with Float32: {time_float:.4f} seconds")

# Run object benchmark
time_obj = benchmark_catboost(X_obj, y, is_object=True)
print(f"Time with Object array: {time_obj:.4f} seconds")

print(f"\nSlowdown Factor: {time_obj / time_float:.2f}x slower")

Starting Benchmark...
Time with Float32: 0.3877 seconds
Time with Object array: 0.3155 seconds

Slowdown Factor: 0.81x slower
